# =========================================================
# CATBOOST MODEL — HDB RESALE PRICE PREDICTION
# =========================================================

In [30]:
# =========================================================
# 0. IMPORT LIBRARIES
# =========================================================
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, r2_score

## Step 1: Load and Clean Data

Three sequential cleaning operations:

1. **Drop nulls** — rows with any missing value are removed (primarily affects amenity distance columns where geocoding returned no result)
2. **Drop duplicates**

In [ ]:
file_path = r"merged_data\[FINAL]hdb_with_amenities_macro_pre2026.csv"
df_raw = pd.read_csv(file_path)
print(f"Initial shape: {df_raw.shape}")

df = df_raw.dropna()
print(f"After dropping nulls: {df.shape} (dropped {len(df_raw) - len(df)})")

# Remove duplicate rows to avoid repeated transactions affecting model training
n_before = len(df)
df = df.drop_duplicates()
print(f"After dropping duplicates: {df.shape} (dropped {n_before - len(df)})")

df['log_resale_price_real'] = np.log(df['resale_price_real'])  # Log-transform target — preserves resale_price_real for metric computation

Initial shape: (134479, 37)
After dropping nulls: (96863, 37) (dropped 37616)
After dropping duplicates: (96803, 37) (dropped 60)


## Step 2: Stratified Train/Validation Split (80/20)

**80/20 ratio:** With ~97,000 transactions, yields ~77,000 training and ~19,000 validation rows. The training set is large enough for stable estimation; the validation set is large enough for statistically reliable RMSE and linlin loss estimates.

**Stratification key: `town + flat_type + year`** — three reasons:
1. **Town** — 26 HDB towns with substantially different price levels; random sampling could over-represent expensive towns in training.
2. **Flat type** — Highly imbalanced (4 ROOM ~42% vs 2 ROOM ~2%); stratifying prevents minority types being underrepresented.
3. **Year** — Despite RPI adjustment, residual structural differences exist across years (post-COVID demand surge, cooling measures); ensures proportional representation of every year in both sets.

In [32]:
# =========================================================
# STEP 2: STRATIFIED OUTER TRAIN / VALIDATION SPLIT (80/20)
# from development set only (2021–2025)
# Purpose:
# - dev_train_df: used for model development
# - model_val_df: held-out validation set for model comparison
# =========================================================

df["strat_key"] = (
    df["town"].astype(str) + "_" +
    df["flat_type"].astype(str) + "_" +
    df["year"].astype(str)
)

# Remove singleton strata because stratified split requires at least 2 rows per stratum
strat_counts = df["strat_key"].value_counts()
valid_keys = strat_counts[strat_counts >= 2].index

n_before = len(df)
df = df[df["strat_key"].isin(valid_keys)].copy()
print(f"Dropped {n_before - len(df)} rows with singleton strat_key combinations. Remaining: {len(df):,}")

dev_train_df, model_val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["strat_key"]
)

print(f"Development train size: {len(dev_train_df):,}")
print(f"Model validation size: {len(model_val_df):,}")

print("\nYear distribution (%):")
dev_year = dev_train_df["year"].value_counts(normalize=True).sort_index().rename("Dev Train %")
model_val_year = model_val_df["year"].value_counts(normalize=True).sort_index().rename("Model Val %")
year_dist = pd.concat([dev_year, model_val_year], axis=1)
print(year_dist.map(lambda x: f"{x:.2%}"))

print("\nFlat type distribution (%):")
dev_flat = dev_train_df["flat_type"].value_counts(normalize=True).rename("Dev Train %")
model_val_flat = model_val_df["flat_type"].value_counts(normalize=True).rename("Model Val %")
flat_dist = pd.concat([dev_flat, model_val_flat], axis=1)
print(flat_dist.map(lambda x: f"{x:.2%}"))

Dropped 11 rows with singleton strat_key combinations. Remaining: 96,792
Development train size: 77,433
Model validation size: 19,359

Year distribution (%):
     Dev Train % Model Val %
year                        
2021      21.92%      21.91%
2022      19.94%      19.99%
2023      18.77%      18.77%
2024      20.62%      20.62%
2025      18.75%      18.71%

Flat type distribution (%):
                 Dev Train % Model Val %
flat_type                               
4 ROOM                41.84%      41.85%
3 ROOM                25.54%      25.52%
5 ROOM                23.58%      23.58%
EXECUTIVE              6.75%       6.77%
2 ROOM                 2.22%       2.21%
1 ROOM                 0.04%       0.04%
MULTI-GENERATION       0.03%       0.03%


## Step 3: Feature Preparation

We prepare the feature matrices and target variables for model training.

- **Numerical features** include proximity measures (e.g., distance to MRT, hawker centres) and lease characteristics  
- **Categorical features** include flat type, town, and floor category  

The target variable is the **log-transformed real resale price**, which helps:
- Stabilise variance
- Improve model performance
- Reduce skewness in price distribution  

We also retain the original (non-log) resale price for evaluation in dollar terms.

Categorical features are explicitly passed to CatBoost, which handles encoding internally.

In [33]:
# =========================================================
# STEP 3: PREPARE FEATURES FOR CATBOOST
#
# continuous_features = [
#   remaining_lease_years, nearest_train_dist_m, dist_nearest_hawker_m,
#   dist_nearest_primary_m, num_primary_1km, dist_nearest_park_m, num_parks_1km,
#   dist_nearest_sportsg_m, dist_nearest_mall_m, dist_nearest_healthcare_m
# ]
# categorical_features = [flat_type, town, floor_category]
#
# Excluded:
# - floor_area_sqm
# - dist_cbd_m
# =========================================================
feature_cols = [
    "remaining_lease_years",
    "nearest_train_dist_m",
    "dist_nearest_hawker_m",
    "dist_nearest_primary_m",
    "num_primary_1km",
    "dist_nearest_park_m",
    "num_parks_1km",
    "dist_nearest_sportsg_m",
    "dist_nearest_mall_m",
    "dist_nearest_healthcare_m",
    "flat_type",
    "town",
    "floor_category"
]

categorical_features = ["flat_type", "town", "floor_category"]

target = "log_resale_price_real"

# Outer validation set for model comparison
X_model_val = model_val_df[feature_cols].copy()
y_model_val_log = model_val_df[target].copy()
y_model_val_actual = model_val_df["resale_price_real"].copy()

print("X_model_val shape:", X_model_val.shape)
print("Number of features:", len(feature_cols))
print("Categorical features:", categorical_features)

X_model_val shape: (19359, 13)
Number of features: 13
Categorical features: ['flat_type', 'town', 'floor_category']


In [34]:
# =========================================================
# STEP 4: INNER SPLIT FOR EARLY STOPPING
# Purpose:
# - fit_train_df: actual training data for CatBoost
# - early_stop_val_df: validation data used only for early stopping
# =========================================================

fit_train_df, early_stop_val_df = train_test_split(
    dev_train_df,
    test_size=0.2,
    random_state=42,
    stratify=dev_train_df["flat_type"]
)

print(f"Fit train size: {len(fit_train_df):,}")
print(f"Early stopping validation size: {len(early_stop_val_df):,}")

X_fit_train = fit_train_df[feature_cols].copy()
y_fit_train_log = fit_train_df[target].copy()

X_early_stop = early_stop_val_df[feature_cols].copy()
y_early_stop_log = early_stop_val_df[target].copy()

cat_feature_indices = [X_fit_train.columns.get_loc(col) for col in categorical_features]

print("X_fit_train shape:", X_fit_train.shape)
print("X_early_stop shape:", X_early_stop.shape)
print("Categorical feature indices:", cat_feature_indices)

Fit train size: 61,946
Early stopping validation size: 15,487
X_fit_train shape: (61946, 13)
X_early_stop shape: (15487, 13)
Categorical feature indices: [10, 11, 12]


## Step 5: Model Training (CatBoost)

We train a **CatBoost Regressor**, which is well-suited for tabular data and handles categorical variables natively.

Key settings:
- Loss function: RMSE  
- Early stopping: stops training if validation performance does not improve  
- Validation set: used to monitor performance and prevent overfitting  

The model is trained on the training set and evaluated on the validation set during training.  
The best iteration (based on validation performance) is automatically selected.

In [39]:
# =========================================================
# STEP 5: FIT CATBOOST MODEL
# =========================================================

cat_model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=3000,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=5,
    random_seed=42,
    early_stopping_rounds=100,
    verbose=200
)

cat_model.fit(
    X_fit_train,
    y_fit_train_log,
    cat_features=cat_feature_indices,
    eval_set=(X_early_stop, y_early_stop_log),
    use_best_model=True
)

print("Best iteration:", cat_model.get_best_iteration())
print("Best validation score:", cat_model.get_best_score())

0:	learn: 0.3069507	test: 0.3049792	best: 0.3049792 (0)	total: 100ms	remaining: 4m 59s
200:	learn: 0.0806058	test: 0.0790533	best: 0.0790533 (200)	total: 19.8s	remaining: 4m 35s
400:	learn: 0.0685912	test: 0.0677515	best: 0.0677515 (400)	total: 40.3s	remaining: 4m 21s
600:	learn: 0.0639023	test: 0.0638023	best: 0.0638023 (600)	total: 1m 1s	remaining: 4m 3s
800:	learn: 0.0613260	test: 0.0617638	best: 0.0617638 (800)	total: 1m 20s	remaining: 3m 41s
1000:	learn: 0.0592992	test: 0.0602469	best: 0.0602469 (1000)	total: 1m 40s	remaining: 3m 20s
1200:	learn: 0.0578898	test: 0.0592863	best: 0.0592863 (1200)	total: 1m 59s	remaining: 2m 58s
1400:	learn: 0.0567124	test: 0.0585348	best: 0.0585348 (1400)	total: 2m 18s	remaining: 2m 37s
1600:	learn: 0.0556995	test: 0.0579317	best: 0.0579317 (1600)	total: 2m 36s	remaining: 2m 16s
1800:	learn: 0.0548856	test: 0.0574651	best: 0.0574651 (1800)	total: 2m 56s	remaining: 1m 57s
2000:	learn: 0.0541001	test: 0.0570564	best: 0.0570564 (2000)	total: 3m 16s	rem

## Step 6: Generating Predictions

After training, predictions are generated for:

- **Training set** (optional, for diagnostics)  
- **Validation set** (for model selection)  

Since the model is trained on log-transformed prices, predictions are exponentiated to obtain values in actual dollar terms.

In [40]:
# =========================================================
# STEP 6: GENERATE PREDICTIONS
# Predictions made in log space, then converted back to SGD
# =========================================================

# Predictions in log space
y_model_val_pred_log = cat_model.predict(X_model_val)

# Convert back to dollar space
y_model_val_pred = np.exp(y_model_val_pred_log)

# Optional: predictions on fit train set for diagnostics
y_fit_train_pred_log = cat_model.predict(X_fit_train)
y_fit_train_pred = np.exp(y_fit_train_pred_log)
y_fit_train_actual = fit_train_df["resale_price_real"].copy()

## Step 7: Model Evaluation

We evaluate model performance using three metrics:

1. **RMSE (Root Mean Squared Error)**  
   - Measures prediction accuracy in dollar terms  
   - Penalises large errors more heavily  

2. **R² (Coefficient of Determination)**  
   - Measures proportion of variance explained by the model  

3. **Lin-Lin Loss (Asymmetric Loss)**  
   - Penalises **underprediction more heavily than overprediction**  
   - Reflects real-world risk (e.g., underestimating housing prices may lead to budget constraints)  

Evaluation is conducted in two stages:

- **Validation set** → used for model tuning and selection  
- **Test set (2026)** → used for final, unbiased performance evaluation  

The test set is not used during model development to ensure a fair assessment of generalisation performance.

In [41]:
# =========================================================
# STEP 7: EVALUATION
# Metrics:
# - RMSE
# - asymmetric Lin-Lin loss
# - 80% interval coverage (using conformal interval instead of OLS formula)
# =========================================================
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.array(y_true) - np.array(y_pred)) ** 2))

def linlin_loss(y_true, y_pred, underpredict_weight=2.0):
    """
    Asymmetric linear loss penalising underprediction more heavily than overprediction.
    underpredict_weight=2.0 means predicted < actual is penalised at 2x.
    Overprediction is penalised at 1x.
    """
    errors = np.array(y_true) - np.array(y_pred)  # positive = underprediction
    loss = np.where(errors > 0, underpredict_weight * errors, -errors)
    return np.mean(loss)

model_val_rmse = rmse(y_model_val_actual, y_model_val_pred)
model_val_linlin = linlin_loss(y_model_val_actual, y_model_val_pred, underpredict_weight=2.0)
model_val_r2 = r2_score(y_model_val_actual, y_model_val_pred)

print("=== CATBOOST MODEL EVALUATION (OUTER VALIDATION SET) ===")
print(f"RMSE:         ${model_val_rmse:,.0f}")
print(f"R²:           {model_val_r2:.3f}")
print(f"Linlin Loss:  ${model_val_linlin:,.0f} (underpredict weight = 2.0)")

=== CATBOOST MODEL EVALUATION (OUTER VALIDATION SET) ===
RMSE:         $40,458
R²:           0.965
Linlin Loss:  $42,879 (underpredict weight = 2.0)
